In [2]:
import numpy as np
import pandas as pd

In [3]:
def virtual_coord(x1, x2, x3):
    """
    Returns the virtual coordinate (or 4th point) for a TIP4P water using the following alogrithm:
    x4 = x1 + a*(x2-x1) + b*(x3-x1)
    where a,b = 0.128012065

    Refer to OPLSAA TIP4P itp file in GROMACS for more information

    Author: Gabe Miles
    """
    a = b = 0.128012065
    return x1 + a * (x2 -x1) + b * (x3 - x1)

In [4]:
from random import randint
from scipy.spatial.transform import Rotation as R
                                            
def rotate_hydrogens(hydrate):
    """
    This function randomizes the orientation of THF in space. It does this with a transform matrix
    provided by the scipy Rotation package. By using this package, THF can be rotated around it's
    center in 3D space, and not just around an axis. This funtion also provides random rotations,
    with a random number of degrees in each direction.

    For more information on the theory, visit:
    https://stackoverflow.com/questions/14607640/rotating-a-vector-in-3d-space

    Input: THF dataframe that includes columns for 'X', 'Y', and 'Z'

    Output: Identical THF dataframe with updated coordinates given a random rotation in 3D space

    Author: Gabe Miles
    """
    rot_hydrate = pd.DataFrame(columns=['ELEMENT', 'X', 'Y', 'Z', 'RESIDUE_NUM', 'MOL'])
    for i in range(1,hydrate['RESIDUE_NUM'].iloc[-1]):

        water = hydrate.loc[hydrate['RESIDUE_NUM'] == i].copy()
        # Center oxygen on (0,0,0)
        center = water[['X','Y','Z']].loc[water['ELEMENT'] == 'OW'].values[0]
        water[['X','Y','Z']] = water[['X','Y','Z']].values - center
        
        # Notate HW and MW points together a random distance
        # This maintains the proper bond angles
        x,y,z = randint(0,360), randint(0,360), randint(0,360)
        r = R.from_euler('zyx', [x,y,z], degrees=True)
        new_xyz = (r.as_matrix() @ water[['X','Y','Z']].values.T).T
        rot_water = pd.DataFrame({'ELEMENT': water['ELEMENT'],
                        'X': new_xyz[:,0],
                        'Y': new_xyz[:,1],
                        'Z': new_xyz[:,2],
                        'RESIDUE_NUM': water['RESIDUE_NUM'],
                        'MOL': water['MOL'],
                        'CHARGE': water['CHARGE']})
                        
        # Move water center back to where it should be, add water to new hydrate
        rot_water[['X','Y','Z']] = rot_water[['X','Y','Z']].values + center
        rot_hydrate = pd.concat([rot_hydrate, rot_water], axis=0, ignore_index=True)

    return rot_hydrate

In [5]:
def dipole(hydrate):
    x_dipole = (hydrate['X'] * hydrate['CHARGE']).sum()
    y_dipole = (hydrate['Y'] * hydrate['CHARGE']).sum()
    z_dipole = (hydrate['Z'] * hydrate['CHARGE']).sum()
    return x_dipole + y_dipole + z_dipole


Create the hydrate unit cell for manipulation

In [6]:
hydrate=pd.read_csv('input/s2-hydrate.csv',index_col=0)

In [7]:
hydrate_ox=pd.DataFrame()
hydrate_ox[['X','Y','Z']]=hydrate[['OX','OY','OZ']]
hydrate_ox.insert(0,'ELEMENT','OW')
hydrate_hy=pd.DataFrame()
hydrate_hy[['X','Y','Z']]=hydrate[['H1X','H1Y','H1Z']]
hydrate_hy.insert(0, 'ELEMENT', 'HW1')
hydrate_hz=pd.DataFrame()
hydrate_hz[['X','Y','Z']]=hydrate[['H2X','H2Y','H2Z']]
hydrate_hz.insert(0, 'ELEMENT', 'HW2')

mw_x_coord = virtual_coord(hydrate_ox['X'], hydrate_hy['X'], hydrate_hz['X'])
mw_y_coord = virtual_coord(hydrate_ox['Y'], hydrate_hy['Y'], hydrate_hz['Y'])
mw_z_coord = virtual_coord(hydrate_ox['Z'], hydrate_hy['Z'], hydrate_hz['Z'])

hydrate_mw = pd.DataFrame({'X': mw_x_coord,
                           'Y': mw_y_coord,
                           'Z': mw_z_coord})
hydrate_mw.insert(0,'ELEMENT','MW')

In [8]:
hydrate_total = pd.DataFrame(columns=['ELEMENT', 'X', 'Y', 'Z', 'RESIDUE_NUM'])
residue_num = 1
for row in range(len(hydrate_ox.index)):
    water = pd.concat([hydrate_ox.iloc[row], hydrate_hy.iloc[row], hydrate_hz.iloc[row], hydrate_mw.iloc[row]],
                       axis=1, ignore_index=True).T   
    water['RESIDUE_NUM'] = residue_num
    hydrate_total = pd.concat([hydrate_total, water], axis=0, ignore_index=True)
    residue_num += 1
hydrate_total['MOL'] = 'HOH'
hydrate_total

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,MW,6.021258,6.023186,5.865343,1,HOH
4,OW,5.32179,5.32179,8.5662,2,HOH
...,...,...,...,...,...,...
539,MW,4.327824,12.992666,13.143288,135,HOH
540,OW,12.98252,4.31665,12.9933,136,HOH
541,HW1,12.44593,3.72957,13.52588,136,HOH
542,HW2,13.51643,3.73198,12.4554,136,HOH


In [9]:
water = hydrate_total.loc[hydrate_total['RESIDUE_NUM'] == 1].copy()
water


,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,MW,6.021258,6.023186,5.865343,1,HOH


In [10]:
water[['X','Y','Z']] = water[['X','Y','Z']].values - water[['X','Y','Z']].loc[water['ELEMENT'] == 'OW'].values[0]
water
# water[['X','Y','Z']].values - water[['X','Y','Z']].loc[water['ELEMENT'] == 'OW'].values[0]


,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,0.0,0.0,0.0,1,HOH
1,HW1,-0.16489,0.91305,-0.23532,1,HOH
2,HW2,0.91785,-0.14503,-0.22969,1,HOH
3,MW,0.096388,0.098316,-0.059527,1,HOH


In [11]:
#Make the box 3 x 3
cell_size = 17.30
num_cells = 3
coord_offset = [[-cell_size, 0, cell_size],[0,0,cell_size],[cell_size, 0, cell_size],
                [-cell_size, 0, 0],[cell_size, 0, 0],
                [-cell_size, 0, -cell_size],[0, 0, -cell_size],[cell_size, 0, -cell_size]]

In [12]:
# Create 3x3 plane crystal
hydrate_3_plane = hydrate_total.copy()
for offset in coord_offset:
    # For hydrate
    residue_num = hydrate_3_plane['RESIDUE_NUM'].iloc[len(hydrate_3_plane.index) - 1]
    hydrate_unit = hydrate_total.copy()
    hydrate_unit[['X','Y','Z']] = hydrate_unit[['X','Y','Z']] + offset
    hydrate_unit['RESIDUE_NUM'] = hydrate_unit['RESIDUE_NUM'] + residue_num
    hydrate_3_plane = pd.concat([hydrate_3_plane, hydrate_unit], axis=0, ignore_index=True)


In [13]:
# Stack 3x3x3 crystal
hydrate_3_3 = hydrate_3_plane.copy()
for i in range(1, num_cells):
    residue_num = hydrate_3_3['RESIDUE_NUM'].iloc[len(hydrate_3_3.index) - 1]
    hydrate_unit = hydrate_3_plane.copy()
    hydrate_unit['Y'] = hydrate_unit['Y'] + (cell_size * i)
    hydrate_unit['RESIDUE_NUM'] = hydrate_unit['RESIDUE_NUM'] + residue_num
    hydrate_3_3 = pd.concat([hydrate_3_3, hydrate_unit], axis=0, ignore_index=True)
hydrate_3_3

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,MW,6.021258,6.023186,5.865343,1,HOH
4,OW,5.32179,5.32179,8.5662,2,HOH
...,...,...,...,...,...,...
14683,MW,21.627824,47.592666,-4.156712,3671,HOH
14684,OW,30.28252,38.91665,-4.3067,3672,HOH
14685,HW1,29.74593,38.32957,-3.77412,3672,HOH
14686,HW2,30.81643,38.33198,-4.8446,3672,HOH


In [14]:
hydrate_3_3['CHARGE'] = 0.
hydrate_3_3['CHARGE'].loc[hydrate_3_3['ELEMENT'] == 'HW1'] = 0.52 
hydrate_3_3['CHARGE'].loc[hydrate_3_3['ELEMENT'] == 'HW2'] = 0.52  
hydrate_3_3['CHARGE'].loc[hydrate_3_3['ELEMENT'] == 'MW'] = -1.04 

/tmp/ipykernel_40089/3273938020.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hydrate_3_3['CHARGE'].loc[hydrate_3_3['ELEMENT'] == 'HW1'] = 0.52
/tmp/ipykernel_40089/3273938020.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hydrate_3_3['CHARGE'].loc[hydrate_3_3['ELEMENT'] == 'HW2'] = 0.52
/tmp/ipykernel_40089/3273938020.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hydrate_3_3['CHARGE'].loc[hydrate_3_3['ELEMENT'] == 'MW

So this is my 3x3x3 unit cell hydrate w/o any proton disorder (or just the original proton posiitions). Now what I want to do is mix up the proton positions and then calculate the dipole moment for the box. The goal is for the dipole moment to be 0, or at least close to 0.

My understanding also is that I am just randomly moving the protons around the oxygen atom. I want to restrain the bond angle to maintain the waters like stability. I can do this by moving the protons at the same time around the oxygen atom. Because I am also using tip4p water, I will need to calculate the virtual point before I calculate the dipole since it moves the oxygen charge closer to the hydrogens. 

So,

1) Randomly rotate protons around the oxygen atoms (I can use my ratation algorithm from randomized THF)
2) Calculate the virtual point for water
3) Calculate the dipole moment for the system
4) If below cutoff, write to pdb. If not, repeat.

And that is it. Seems pretty straightforward.

My concern right now is how long an algorithm is going to take that has to randomly rotate things and hope for the best. I think it's a good start, but there is probably a measure I can make that would allow me to prefer or constrain moves to funnel towards a minimum. But, randomly moving things should work in this case.

In [15]:
# (1) Randomly rotate protons
new_hydrate = rotate_hydrogens(hydrate_3_3)

First iteration took 1:10.3, not good. Need to speed things up. The double for loop is the thing killing me at the moment. So I need to figure out a way to condense that code down so something more succint.

In [16]:
f = open("output/rot_hydrate.pdb", "w")

f.write("CRYST1   52.000   52.000   52.000  90.00  90.00  90.00 P 1           1\n")
for i in range(len(new_hydrate.index)):
    #Collect PDB variables
    id = str(i)
    chainid = 'A'
    atom = new_hydrate['ELEMENT'].iloc[i]
    resname = new_hydrate['MOL'].iloc[i]
    resnum = new_hydrate['RESIDUE_NUM'].iloc[i]
    x = new_hydrate['X'].iloc[i]
    y = new_hydrate['Y'].iloc[i]
    z = new_hydrate['Z'].iloc[i]

    # Create PDB entry
    line = "{:6s}{:5d} {:^4s} {:3s} {:1s}{:4d}    {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}\n".format(
           "ATOM", i+1, atom, resname, chainid, resnum, x, y, z, 0., 0.)
    f.write(line)

f.close()

With visualization, the system looks great. In fact, a lot better than it did originally. Like it looks mixed versus a bunch of unit cells stacked together. 

So that is part (1). Now to calculated the dipole moment. I've done it before, so it should not be a big deal to implement.

In [17]:
np.abs(dipole(new_hydrate))

25.22550841886319

In [ ]:
i = 0
while np.abs(dipole(new_hydrate)) >= 0.05:
    new_hydrate = rotate_hydrogens(hydrate_3_3)
    print("Round {}: dipole moment = {}".format(i, np.abs(dipole(new_hydrate))))
    i += 1